# Visión por Computadora — Análisis Facial con DeepFace

Este notebook demuestra el uso de DeepFace para:
- Detección y análisis de rostros (emoción, edad, género)
- Verificación de identidad entre dos imágenes
- Visualización de resultados

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO
import os

from deepface import DeepFace

## 1. Descarga de imágenes de muestra

In [ ]:
os.makedirs('img', exist_ok=True)

# Imágenes de dominio público (faces del dataset de DeepFace)
imagenes = {
    'img/persona1.jpg': 'https://raw.githubusercontent.com/serengil/deepface/master/tests/dataset/img1.jpg',
    'img/persona2.jpg': 'https://raw.githubusercontent.com/serengil/deepface/master/tests/dataset/img2.jpg',
    'img/persona3.jpg': 'https://raw.githubusercontent.com/serengil/deepface/master/tests/dataset/img3.jpg',
}

for ruta, url in imagenes.items():
    if not os.path.exists(ruta):
        resp = requests.get(url)
        with open(ruta, 'wb') as f:
            f.write(resp.content)
        print(f'Descargada: {ruta}')
    else:
        print(f'Ya existe: {ruta}')

## 2. Visualización de las imágenes

In [ ]:
rutas = list(imagenes.keys())
fig, axes = plt.subplots(1, len(rutas), figsize=(12, 4))
for ax, ruta in zip(axes, rutas):
    img = Image.open(ruta)
    ax.imshow(img)
    ax.set_title(os.path.basename(ruta))
    ax.axis('off')
plt.suptitle('Imágenes de muestra', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Análisis facial: emoción, edad y género

In [ ]:
resultados = []
for ruta in rutas:
    analisis = DeepFace.analyze(
        img_path=ruta,
        actions=['emotion', 'age', 'gender'],
        enforce_detection=False
    )
    r = analisis[0]
    resultados.append({
        'imagen': os.path.basename(ruta),
        'emocion_dominante': r['dominant_emotion'],
        'edad_estimada': r['age'],
        'genero': r['dominant_gender'],
    })

df_resultados = pd.DataFrame(resultados)
df_resultados

## 4. Visualización de emociones por imagen

In [ ]:
fig, axes = plt.subplots(1, len(rutas), figsize=(15, 4))

for ax, ruta in zip(axes, rutas):
    analisis = DeepFace.analyze(
        img_path=ruta,
        actions=['emotion'],
        enforce_detection=False
    )
    emociones = analisis[0]['emotion']
    nombres = list(emociones.keys())
    valores = list(emociones.values())
    colores = ['#e74c3c' if n == max(emociones, key=emociones.get) else '#3498db' for n in nombres]
    ax.barh(nombres, valores, color=colores)
    ax.set_title(os.path.basename(ruta))
    ax.set_xlabel('Probabilidad (%)')
    ax.set_xlim(0, 100)

plt.suptitle('Distribución de emociones por imagen', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Verificación de identidad entre dos imágenes

In [ ]:
pares = [
    ('img/persona1.jpg', 'img/persona2.jpg'),
    ('img/persona1.jpg', 'img/persona3.jpg'),
]

filas = []
for img1, img2 in pares:
    resultado = DeepFace.verify(
        img1_path=img1,
        img2_path=img2,
        enforce_detection=False
    )
    filas.append({
        'imagen_1': os.path.basename(img1),
        'imagen_2': os.path.basename(img2),
        'misma_persona': resultado['verified'],
        'distancia': round(resultado['distance'], 4),
        'umbral': round(resultado['threshold'], 4),
    })

pd.DataFrame(filas)

## 6. Detección de rostros con bounding boxes

In [ ]:
ruta_demo = 'img/persona1.jpg'
img_array = np.array(Image.open(ruta_demo))

rostros = DeepFace.extract_faces(
    img_path=ruta_demo,
    enforce_detection=False
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_array)

for rostro in rostros:
    region = rostro['facial_area']
    x, y, w, h = region['x'], region['y'], region['w'], region['h']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 5, f"conf: {rostro['confidence']:.2f}", color='lime', fontsize=9)

ax.set_title('Detección de rostros')
ax.axis('off')
plt.tight_layout()
plt.show()